In [30]:
# IMPORTSSSS
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout, Concatenate, RepeatVector, TimeDistributed

In [31]:

# Preprocess blocks
def analyze_stroke_data(csv_filepath):
    # 1. Load the data
    df = pd.read_csv(csv_filepath)
    
    # 2. Calculate time differences between rows (Delta Time / dt)
    df['dt'] = df['time'].diff().fillna(0)
    
    # 3. Calculate distance between points (Delta Distance using Pythagorean theorem)
    df['dx'] = df['x'].diff().fillna(0)
    df['dy'] = df['y'].diff().fillna(0)
    
    # 4. Calculate Velocity (Distance / Time)
    # np.where prevents division-by-zero errors if two events fire at the exact same millisecond
    df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2) 
    df['velocity'] = np.where(df['dt'] > 0, df['distance'] / df['dt'], 0)
    
    # Calculate "Writing Duration" 
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 1
    writing_duration = df['dt'].where(df['touching'] == 1, 0).sum()

    # Calculate "In-Air Pen Duration" (The pause time biomarker)
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 0
    in_air_duration = df['dt'].where(df['touching'] == 0, 0).sum()
    
    # TODO 3: Print the results!
    print(f"--- Analysis for: {csv_filepath} ---")
    print(f"Total Writing Duration : {writing_duration} ms")
    print(f"Total In-Air Pauses    : {in_air_duration} ms")
    print(f"Average Pen Velocity   : {df['velocity'].mean():.2f} px/ms")
    print("-" * 40)
    
    return df


In [32]:

def load_and_pad_data(data_dir):
    sequences = []
    latencies = []
    labels = []
    
    for filename in os.listdir(data_dir):
        if not filename.endswith('.csv'):
            continue
            
        # Grab the label (1 for dyslexia, 0 for normal)
        label = 1 if filename.startswith("dyslexia") else 0
        smart_labels = np.zeros((MAX_TIMESTEPS, 1))
        
        filepath = os.path.join(data_dir, filename)
       
        # loads the CSV and calculates the velocity/distances.
        df = analyze_stroke_data(filepath)
        
        actual_length = min(len(df), MAX_TIMESTEPS)
        
        if label == 1: # ONLY apply this logic if the file is from a Dyslexic sample!
            
            # Figure out what "slow" means for this specific patient
            # grab the bottom 25% of their writing speed to define a "stutter"
            moving_data = df[df['velocity'] > 0]['velocity']
            stutter_threshold = np.percentile(moving_data, 25) if len(moving_data) > 0 else 0.5
            
            for i in range(actual_length):
                # Anomaly A: The pen is lifted in the air (Hesitation)
                is_paused = (df['touching'].iloc[i] == 0)
                
                # Anomaly B: The pen is touching, but dragging very slowly (Micro-stutter)
                is_stuttering = (df['touching'].iloc[i] == 1) and (df['velocity'].iloc[i] < stutter_threshold)
                
                # If they are struggling at this exact millisecond, flag it!
                if is_paused or is_stuttering:
                    smart_labels[i] = 1 
                    
        
        if 'latency' in df.columns:
            latency_val = df['latency'].iloc[0]
        else:
            latency_val = 0
            
        # Extract just the features we want
        stroke_data = df[['velocity', 'pressure', 'touching']].values
        
        # Pad or Truncate to MAX_TIMESTEPS (500)
        if len(stroke_data) > MAX_TIMESTEPS:
            stroke_data = stroke_data[:MAX_TIMESTEPS] # Truncate if too long
        else:
            # Pad with zeros if too short
            padding = np.zeros((MAX_TIMESTEPS - len(stroke_data), FEATURES))
            stroke_data = np.vstack((stroke_data, padding))
            
        sequences.append(stroke_data)
        latencies.append(latency_val)
        # labels.append(label)
        # labels.append(np.full((MAX_TIMESTEPS, 1), label))
        labels.append(smart_labels)
        
    return np.array(sequences), np.array(latencies), np.array(labels)


In [37]:
# Hyperparameters
MAX_TIMESTEPS = 500  #  standardize all writing samples to 500 time-steps
FEATURES = 3         #  feed it with 3 features: [velocity, pressure, touching]

def build_model():
    sequence_input = Input(shape=(MAX_TIMESTEPS, FEATURES), name="kinematics")
    # padding='same' ensures the sequence stays exactly 500 steps long
    x = Conv1D(32, kernel_size=15, activation='relu', padding='same')(sequence_input)
    x = Conv1D(64, kernel_size=10, dilation_rate=2, activation='relu', padding='same')(x)
    # Latency INput
    latency_input = Input(shape=(1,), name="latency")
    # Stretch the 1 single latency number into 500 copies
    repeated_latency = RepeatVector(MAX_TIMESTEPS)(latency_input) 
    
    merged = Concatenate()([x, repeated_latency]) 
    
    y = TimeDistributed(Dense(64, activation='relu'))(merged)
    y = Dropout(0.5)(y)
    final_output = TimeDistributed(Dense(1, activation='sigmoid'))(y)
    
    model = Model(inputs=[sequence_input, latency_input], outputs=final_output)
    model.compile(
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
    )
    return model

In [38]:
model = build_model()
model.summary()
print("Loading data...")

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ kinematics          │ (None, 500, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 500, 32)   │      1,472 │ kinematics[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latency             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 500, 64)   │     20,544 │ conv1d_12[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_6     │ (None, 500, 1)    │          0 │ latency[0][0]     │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 500, 65)   │          0 │ conv1d_13[0][0],  │
│ (Concatenate)       │                   │            │ repeat_vector_6[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_12 │ (None, 500, 64)   │      4,224 │ concatenate_6[0]… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 500, 64)   │          0 │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_13 │ (None, 500, 1)    │         65 │ dropout_6[0][0]   │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 26,305 (102.75 KB)

 Trainable params: 26,305 (102.75 KB)

 Non-trainable params: 0 (0.00 B)

Loading data...


In [39]:
# Point this to your data collector folder
X_seq, X_lat, y = load_and_pad_data("datasets/") 
print(f"Data loaded! \nShape of X_seq (timeseries sequence) : {X_seq.shape} \nShape of X_lat (latency) :{X_lat.shape} \nShape of y (labels) : {y.shape}")
    

--- Analysis for: datasets/dyslexia_A_1787797509.csv ---
Total Writing Duration : 2472.6999999955297 ms
Total In-Air Pauses    : 24.80000000447035 ms
Average Pen Velocity   : 0.67 px/ms
----------------------------------------
--- Analysis for: datasets/dyslexia_A_1787797536.csv ---
Total Writing Duration : 9554.400000002235 ms
Total In-Air Pauses    : 67.29999999701977 ms
Average Pen Velocity   : 0.26 px/ms
----------------------------------------
--- Analysis for: datasets/dyslexia_A_1787797518.csv ---
Total Writing Duration : 2955.89999999851 ms
Total In-Air Pauses    : 16.699999999254942 ms
Average Pen Velocity   : 0.37 px/ms
----------------------------------------
--- Analysis for: datasets/dyslexia_A_1787797478.csv ---
Total Writing Duration : 2713.199999999255 ms
Total In-Air Pauses    : 59.600000001490116 ms
Average Pen Velocity   : 0.46 px/ms
----------------------------------------
--- Analysis for: datasets/normal_A_1787797758.csv ---
Total Writing Duration : 1904.199999999

In [ ]:
import numpy as np

# Count how many total 0s and 1s exist in the entire training set 'y'
total_zeros = np.sum(y == 0)
total_ones = np.sum(y == 1)
total_samples = total_zeros + total_ones

# Apply the 'Weight = Total_Samples / (Number_of_Classes * Samples_in_Class)' formula
weight_for_0 = total_samples / (2.0 * total_zeros)
weight_for_1 = total_samples / (2.0 * total_ones)

# 3. Create the exact dictionary
imbalance_weights = {
    0: weight_for_0,
    1: weight_for_1
}

print(f"Calculated Weights -> 0 (normal): {weight_for_0:.2f}, 1 (dyslexic): {weight_for_1:.2f}")

In [ ]:

print("Starting training...")
# history = model.fit(x_train, y_train, epochs=20, validation_split=0.2)
history = model.fit([X_seq, X_lat], y, epochs=20, validation_split=0.2, class_weight=imbalance_weights)
print("Saving the model...")
model.save("models/elkinematicV2.keras")

Starting training...
Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 36s 36s/step - accuracy: 0.7624 - loss: 3.6574 - recall: 0.2056 - val_accuracy: 0.9810 - val_loss: 0.3062 - val_recall: 0.0000e+00
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step - accuracy: 0.7899 - loss: 3.2360 - recall: 0.2056 - val_accuracy: 0.9810 - val_loss: 0.3062 - val_recall: 0.0000e+00
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 0.7924 - loss: 3.1792 - recall: 0.2056 - val_accuracy: 0.9810 - val_loss: 0.3062 - val_recall: 0.0000e+00
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.8114 - loss: 2.8802 - recall: 0.1308 - val_accuracy: 0.9810 - val_loss: 0.3062 - val_recall: 0.0000e+00
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.8217 - loss: 2.7431 - recall: 0.1526 - val_accuracy: 0.9810 - val_loss: 0.3062 - val_recall: 0.0000e+00
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 342ms/step - accuracy: 0.8269 - loss: 2.6493 - recall: 0.1745 - val_accuracy: 0.9810 - val_loss: 0.30